# City temperature and air-quality animation

This notebook demonstrates `sentinel_analysis`'s multisensor acquisition
pipeline (`AnalysisRequest` + `AnalysisWorkflow`) end to end, then turns the
result into something the demo notebooks don't yet show: an **animation**
of how a field evolves over several days, plus **point time series** at a
handful of named locations inside the city.

Two independent runs:

- **Sentinel-3 SLSTR land surface temperature (LST)** - one frame per
  usable overpass.
- **Sentinel-5P TROPOMI NO2** - one frame per usable overpass.

Both requests target the same `CitySpec`, so both results land on the same
projected analysis grid - the point locations and axes are directly
comparable between the two animations.

Sentinel-3 and Sentinel-5P are overpass sensors: frames are real, irregular
satellite acquisitions, not an interpolated continuous record. Some
requested days may have no usable (cloud-free / QA-passing) observation at
all - this notebook reports exactly which acquisition timestamps ended up
in each animation rather than pretending otherwise.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
import xarray as xr
from IPython.display import Image, display
from pyproj import Transformer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sentinel_analysis import (
    AnalysisRequest,
    AnalysisResult,
    AnalysisWorkflow,
    ClientConfig,
    get_city,
)

## 1. Configuration

Every parameter is overridable via environment variables so this notebook
can be re-run for a different city, period, or frame budget without
editing code - the same convention `berlin_multisensor_downscaling.ipynb`
uses. `RUN_LIVE=0` (the default) never touches the network: it reuses a
previously saved cube if one exists under `OUTPUT_DIR`, so opening this
notebook is always safe.

`MAX_PRODUCTS_*` bounds how many real acquisitions get downloaded and
processed - each Sentinel-3 LST frame involves a real download plus a
footprint-based regrid (a few minutes each on a laptop), so keep this
modest for a first run.

In [ ]:
RUN_LIVE = os.getenv('SENTINEL_NOTEBOOK_LIVE', '0') == '1'
LOAD_EXISTING = os.getenv('SENTINEL_NOTEBOOK_LOAD_EXISTING', '1') == '1'

CITY = get_city(os.getenv('SENTINEL_NOTEBOOK_CITY', 'berlin'))
START_DATE = os.getenv('SENTINEL_NOTEBOOK_START', '2026-08-15')
END_DATE = os.getenv('SENTINEL_NOTEBOOK_END', '2026-08-28')
RESOLUTION_M = int(os.getenv('SENTINEL_NOTEBOOK_RESOLUTION_M', '1000'))
MAX_PRODUCTS_S3 = int(os.getenv('SENTINEL_NOTEBOOK_MAX_PRODUCTS_S3', '5'))
MAX_PRODUCTS_S5P = int(os.getenv('SENTINEL_NOTEBOOK_MAX_PRODUCTS_S5P', '8'))
NO2_VARIABLE = os.getenv('SENTINEL_NOTEBOOK_S5P_GAS', 'NO2')

OUTPUT_DIR = Path(os.getenv('SENTINEL_NOTEBOOK_OUTPUT', str(PROJECT_ROOT / 'output' / 'city-temperature-animation')))
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = PROJECT_ROOT / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# A handful of named points inside the AOI to track individually, in
# addition to the full-grid animation - e.g. a city-centre pixel versus an
# outskirt pixel can show a real urban heat island contrast.
POINTS_LONLAT = {
    'City centre': (13.405, 52.520),
    'North edge': (13.30, 52.565),
    'South edge': (13.45, 52.415),
}

print({
    'live': RUN_LIVE,
    'city': CITY.name,
    'period': (START_DATE, END_DATE),
    'resolution_m': RESOLUTION_M,
    'max_products': {'sentinel3': MAX_PRODUCTS_S3, 'sentinel5p': MAX_PRODUCTS_S5P},
    'output': str(OUTPUT_DIR),
})

## 2. Shared helpers

Both sensors end up regridded onto the *same* projected `CitySpec.crs`
(`AnalysisRequest.for_city` builds one predictor grid per request, always
at the same city + resolution, and `AnalysisWorkflow` reprojects Sentinel-5P
onto it too) - so one pair of helpers covers both animations:

- `point_series` looks up the nearest grid cell to a lon/lat point.
- `animate_cube` renders a `time, y, x` DataArray as a saved GIF, with the
  named points marked on the map.

In [ ]:
_LONLAT_TO_CITY_CRS = Transformer.from_crs('EPSG:4326', CITY.crs, always_xy=True)


def project_points(points_lonlat):
    """lon/lat -> the city's projected CRS, matching every returned cube."""
    projected = {}
    for label, (lon, lat) in points_lonlat.items():
        x, y = _LONLAT_TO_CITY_CRS.transform(lon, lat)
        projected[label] = (float(x), float(y))
    return projected


def point_series(cube, variable, points_xy):
    """Nearest-cell time series at each named point."""
    series = {}
    for label, (x, y) in points_xy.items():
        series[label] = cube[variable].sel(x=x, y=y, method='nearest').values
    return series


def plot_point_series(cube, variable, points_xy, *, title, ylabel):
    series = point_series(cube, variable, points_xy)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for label, values in series.items():
        ax.plot(cube.time.values, values, 'o-', label=label)
    ax.set_title(title)
    ax.set_xlabel('Acquisition time (UTC)')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    return series


def animate_cube(cube, variable, *, title, cmap, unit_label, points_xy, filename, seconds_per_frame=1.2):
    """Save a time,y,x DataArray as an animated GIF and return its path."""
    data = cube[variable]
    n_frames = data.sizes['time']
    values = data.values
    finite = np.isfinite(values)
    vmin = float(np.nanmin(values)) if finite.any() else 0.0
    vmax = float(np.nanmax(values)) if finite.any() else 1.0

    x, y = cube['x'].values, cube['y'].values
    extent = [x.min(), x.max(), y.min(), y.max()]

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad('lightgray')

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(
        np.ma.masked_invalid(data.isel(time=0).values),
        origin='upper', extent=extent, cmap=cmap_obj, vmin=vmin, vmax=vmax, aspect='equal',
    )
    fig.colorbar(image, ax=ax, label=unit_label)
    ax.set_xlabel(f'x, {CITY.crs} (m)')
    ax.set_ylabel(f'y, {CITY.crs} (m)')

    if points_xy:
        px = [p[0] for p in points_xy.values()]
        py = [p[1] for p in points_xy.values()]
        ax.scatter(px, py, marker='x', color='black', s=70, linewidths=2, zorder=5)
        for label, (mx, my) in points_xy.items():
            ax.annotate(label, (mx, my), textcoords='offset points', xytext=(6, 6), fontsize=9)

    def update(frame_index):
        image.set_data(np.ma.masked_invalid(data.isel(time=frame_index).values))
        timestamp = np.datetime_as_string(cube.time.values[frame_index], unit='m')
        ax.set_title(f'{title}\n{timestamp} UTC  (frame {frame_index + 1}/{n_frames})')
        return (image,)

    update(0)
    anim = FuncAnimation(fig, update, frames=n_frames, interval=seconds_per_frame * 1000, blit=False)
    anim.save(filename, writer=PillowWriter(fps=1 / seconds_per_frame))
    plt.close(fig)
    return Path(filename)


POINTS_XY = project_points(POINTS_LONLAT)
print('Projected points (city CRS, metres):', POINTS_XY)


def netcdf_safe(dataset):
    """netCDF attributes can't hold Python bools (e.g. grid_l2_lst's
    pixel_footprints_reconstructed) or dict/list/tuple values - coerce them
    to netCDF-safe types before to_netcdf(), same pattern used in
    berlin_multisensor_downscaling.ipynb."""
    def safe_attrs(attrs):
        return {
            key: int(value) if isinstance(value, (bool, np.bool_))
            else str(value) if isinstance(value, (dict, list, tuple))
            else value
            for key, value in attrs.items()
        }
    clean = dataset.copy()
    clean.attrs = safe_attrs(clean.attrs)
    for name in clean.variables:
        clean[name].attrs = safe_attrs(clean[name].attrs)
    return clean

## 3. Sentinel-3 land surface temperature

### A residual-cloud sanity filter

Satellite LST retrievals occasionally pass every documented QC flag
(`exception_flags`, `cloud_mask`) while still being contaminated by
residual/undetected cloud - a well-known, hard-to-fully-flag limitation of
thermal remote sensing, not specific to this pipeline. When it happens the
symptom is a large, spatially *contiguous* patch of implausibly cold
pixels (clouds are cold; a mid-latitude city in August is not).

A per-frame robust statistic (MAD / modified z-score) was tried first and
does **not** work here: because the contamination is one big contiguous
blob rather than scattered speckle, it drags the frame's own median down
with it, so nothing looks like a statistical outlier relative to that
shifted median. What actually works is a plain physical sanity bound -
reject anything colder than a threshold no clear-sky pixel should reach
for this city and season. This is deliberately blunt and dataset-specific,
not a general-purpose algorithm; see the printed counts below for exactly
what it removes.

In [ ]:
def mask_implausible_cold_lst(cube, *, min_plausible_celsius):
    """Reject per-frame LST pixels colder than a physical sanity bound.

    Only ever applied to LST: the same idea would be wrong for a gas
    concentration field like Sentinel-5P NO2, where a spatial outlier can
    be a genuine point-source plume rather than an artifact.
    """
    lst = cube['lst'].copy()
    total_rejected = 0
    for i in range(lst.sizes['time']):
        frame = lst.isel(time=i).values
        cold = np.isfinite(frame) & (frame < min_plausible_celsius)
        total_rejected += int(cold.sum())
        frame[cold] = np.nan
        lst.values[i] = frame
    print(f'Rejected {total_rejected} residual-cloud-contaminated cell(s) '
          f'(< {min_plausible_celsius} degC) across {lst.sizes["time"]} frames')
    out = cube.copy()
    out['lst'] = lst
    return out


MIN_PLAUSIBLE_LST_C = float(os.getenv('SENTINEL_NOTEBOOK_MIN_PLAUSIBLE_LST_C', '10'))

## 4. Sentinel-3 acquisition

`AnalysisRequest.for_city` builds both the AOI and a snapped, projected
`AnalysisGrid` from the `CitySpec` in one call. `AnalysisWorkflow.execute()`
then discovers, downloads, georeferences (each acquisition individually -
see `_combine_sentinel3` in `workflow/runner.py`) and quality-masks every
requested product, converting Kelvin to Celsius on the way out.

In [ ]:
s3_request = AnalysisRequest.for_city(
    CITY, START_DATE, END_DATE,
    sensors=('sentinel3',),
    resolution_m=RESOLUTION_M,
    max_products_per_sensor=MAX_PRODUCTS_S3,
)

s3_cube_path = OUTPUT_DIR / 's3_lst_cube.nc'
if RUN_LIVE:
    s3_result = AnalysisWorkflow(s3_request).execute(
        OUTPUT_DIR / 'sentinel3', config=ClientConfig.from_env(PROJECT_ROOT / '.env'), max_workers=3,
    )
    s3_cube = s3_result.cube
    netcdf_safe(s3_cube).to_netcdf(s3_cube_path)
    print(f'Acquired {s3_cube.sizes["time"]} Sentinel-3 frames; saved to {s3_cube_path}')
elif LOAD_EXISTING and s3_cube_path.exists():
    s3_cube = xr.load_dataset(s3_cube_path)
    print(f'Loaded {s3_cube.sizes["time"]} existing Sentinel-3 frames from {s3_cube_path}')
else:
    s3_cube = None
    print('Live acquisition disabled and no saved Sentinel-3 cube was found. '
          'Set SENTINEL_NOTEBOOK_LIVE=1 and re-run this cell.')

if s3_cube is not None:
    s3_cube = mask_implausible_cold_lst(s3_cube, min_plausible_celsius=MIN_PLAUSIBLE_LST_C)

In [ ]:
if s3_cube is not None:
    print('Grid:', dict(s3_cube.sizes), '| crs:', s3_cube.attrs.get('crs'))
    print('Frame timestamps (UTC):')
    for t in s3_cube.time.values:
        print(' ', np.datetime_as_string(t, unit='m'))

### Point time series

Nearest-cell LST at each named point across every frame - the city-centre
point is expected to run warmer than the edges during the day (urban heat
island), though with only a handful of overpasses this is illustrative,
not a statistically robust comparison.

In [ ]:
if s3_cube is not None:
    plot_point_series(
        s3_cube, 'lst', POINTS_XY,
        title=f'{CITY.name}: Sentinel-3 LST at named points',
        ylabel='LST (degC)',
    )

### Animation

Saved as a GIF (no `ffmpeg` dependency) so it plays back even when this
notebook is reopened without a live kernel.

In [ ]:
if s3_cube is not None:
    s3_gif_path = animate_cube(
        s3_cube, 'lst',
        title=f'{CITY.name} - Sentinel-3 LST',
        cmap='inferno', unit_label='LST (degC)',
        points_xy=POINTS_XY,
        filename=OUTPUT_DIR / 's3_lst_animation.gif',
    )
    print(f'Saved animation: {s3_gif_path}')
    display(Image(filename=str(s3_gif_path)))

## 5. Sentinel-5P NO2

Same pattern, different sensor: `AnalysisWorkflow` reads each TROPOMI
swath, grids it, and reprojects it onto the *same* city grid used above -
so the point locations and axes below line up exactly with the LST
animation.

In [ ]:
s5p_request = AnalysisRequest.for_city(
    CITY, START_DATE, END_DATE,
    sensors=('sentinel5p',),
    variables=(NO2_VARIABLE,),
    resolution_m=RESOLUTION_M,
    max_products_per_sensor=MAX_PRODUCTS_S5P,
)

s5p_cube_path = OUTPUT_DIR / 's5p_no2_cube.nc'
if RUN_LIVE:
    s5p_result = AnalysisWorkflow(s5p_request).execute(
        OUTPUT_DIR / 'sentinel5p', config=ClientConfig.from_env(PROJECT_ROOT / '.env'), max_workers=3,
    )
    s5p_cube = s5p_result.cube
    netcdf_safe(s5p_cube).to_netcdf(s5p_cube_path)
    print(f'Acquired {s5p_cube.sizes["time"]} Sentinel-5P frames; saved to {s5p_cube_path}')
elif LOAD_EXISTING and s5p_cube_path.exists():
    s5p_cube = xr.load_dataset(s5p_cube_path)
    print(f'Loaded {s5p_cube.sizes["time"]} existing Sentinel-5P frames from {s5p_cube_path}')
else:
    s5p_cube = None
    print('Live acquisition disabled and no saved Sentinel-5P cube was found. '
          'Set SENTINEL_NOTEBOOK_LIVE=1 and re-run this cell.')

In [ ]:
if s5p_cube is not None:
    print('Grid:', dict(s5p_cube.sizes), '| crs:', s5p_cube.attrs.get('crs'))
    print('Frame timestamps (UTC):')
    for t in s5p_cube.time.values:
        print(' ', np.datetime_as_string(t, unit='m'))

### Point time series and animation

In [ ]:
if s5p_cube is not None:
    plot_point_series(
        s5p_cube, NO2_VARIABLE, POINTS_XY,
        title=f'{CITY.name}: Sentinel-5P {NO2_VARIABLE} at named points',
        ylabel=f'{NO2_VARIABLE} (mol/m^2)',
    )

In [ ]:
if s5p_cube is not None:
    s5p_gif_path = animate_cube(
        s5p_cube, NO2_VARIABLE,
        title=f'{CITY.name} - Sentinel-5P {NO2_VARIABLE}',
        cmap='viridis', unit_label=f'{NO2_VARIABLE} (mol/m^2)',
        points_xy=POINTS_XY,
        filename=OUTPUT_DIR / 's5p_no2_animation.gif',
    )
    print(f'Saved animation: {s5p_gif_path}')
    display(Image(filename=str(s5p_gif_path)))

## Takeaways

- `AnalysisRequest.for_city` + `AnalysisWorkflow.execute()` is the same
  entry point used by the InSAR-Orchestrator `sentinel-worker` integration
  - this notebook exercises the exact code path a submitted "Multisensor
  Analysis" job runs.
- Both animations reuse one pair of helpers (`animate_cube`,
  `point_series`) because Sentinel-3 and Sentinel-5P land on the same
  projected city grid after `AnalysisWorkflow` harmonizes them - that
  reprojection is what makes the point locations directly comparable
  across sensors.
- Frame counts are capped by `MAX_PRODUCTS_S3` / `MAX_PRODUCTS_S5P`, not by
  calendar days: an overpass sensor does not guarantee one usable
  observation per day, so the printed timestamp list is the ground truth
  for what the animation actually shows, not the requested date range.